In [ ]:
# DANE Z API POBIERANY PRZEZ PYTHON BEZ SPARK'a

In [ ]:
%pip install -q google-cloud-secret-manager

In [ ]:
### PROBA POLACZENIA Z SECRET API

In [ ]:
from google.cloud import secretmanager

In [ ]:
PROJECT_ID = "gcp-pde-498614"
SECRET_ID_key = "alpha-vantage-api-key"
SECRET_ID_email = "sec-email"

client = secretmanager.SecretManagerServiceClient()

# first secret
secret_name_api_key = (
    f"projects/{PROJECT_ID}/"
    f"secrets/{SECRET_ID_key}/"
    f"versions/latest"
)

response = client.access_secret_version(
    request = {"name": secret_name_api_key}
)

API_KEY = response.payload.data.decode("UTF-8")


# second secret

secret_email_sec = (
    f"projects/{PROJECT_ID}/"
    f"secrets/{SECRET_ID_email}/"
    f"versions/latest"
)

response = client.access_secret_version(
    request = {"name": secret_email_sec}
)

API_email = response.payload.data.decode("UTF-8")

In [ ]:
API_KEY
API_email

In [ ]:
import pandas as pd
import requests
import datetime
from zoneinfo import ZoneInfo

In [ ]:
# I IMPORT Z API

In [ ]:
### 1. SEC

In [ ]:
headers = {'User-Agent': API_email}

In [ ]:
companyTickers = requests.get("https://www.sec.gov/files/company_tickers_exchange.json", headers = headers)
companyTickers

In [ ]:
#companyTickers.json()

In [ ]:
companyTickers.json()["data"]
df = pd.DataFrame(companyTickers.json()["data"], columns = ['cik', 'name', 'ticker', 'exchange'])
df["cik"] = df["cik"].astype(str).str.zfill(10)
df.head()

In [ ]:
cik_list = df.head(50)["cik"].to_list()
#cik_list

In [ ]:
df.groupby("name")["ticker"].nunique().sort_values(ascending = False)

In [ ]:
#### 2. SEC - dane spółki

In [ ]:
cik = "0001045810"

In [ ]:
# METADATA DLA OKRESLONEGO CIK
fillingMetadata = requests.get(f'https://data.sec.gov/submissions/CIK{cik}.json', headers = headers)

In [ ]:
allForms = pd.DataFrame.from_dict(fillingMetadata.json()["filings"]["recent"])
allForms.head()

In [ ]:
### 3. XBRL

In [ ]:
#companyFacts = requests.get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json", headers = headers)

In [ ]:
#companyFacts.json()

In [ ]:
### 4. Alpha Vantage API

In [ ]:
TICKER = "NVDA"
TICKER_2 = "AAPL"

In [ ]:
alpha_vantage = requests.get(f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={TICKER}&outputsize=compact&apikey={API_KEY}")
#alpha_vantage.json()

In [ ]:
                                                            # II ZAPIS do warstwy Bronze

In [ ]:
    ### 2.1. SEC metadata

In [ ]:
from google.cloud import storage
import json

In [ ]:
!date

In [ ]:
datetime.datetime.now().astimezone()

In [ ]:
now = str(datetime.datetime.now(ZoneInfo("Europe/Warsaw")))
now

In [ ]:
#companyTickers.json()

In [ ]:
#json.dumps(companyTickers.json())

In [ ]:
storage_client = storage.Client()
bucket = storage_client.get_bucket("project-dev-storage")

In [ ]:
blob = bucket.blob(f"CIK/sec_metadata{str(datetime.datetime.now(ZoneInfo("Europe/Warsaw")))}")
blob.upload_from_string(json.dumps(companyTickers.json()), content_type = "json")

In [ ]:
    # 2.2. SEC dane spółki

In [ ]:
#fillingMetadata.json()

In [ ]:
cik2 = "0000320193"

In [ ]:
fillingMetadata2 = requests.get(f'https://data.sec.gov/submissions/CIK{cik2}.json', headers = headers)

In [ ]:
# KONWERSJA KILKU CIK do wspolnego pliku json
# tutaj nalezy zrobic for loop gdzie bedzie z listy cik_list brac dane dla konkretnych spolek i laczyc je w spolny plik json i ladowac do bronze
wyn = [fillingMetadata.json(), fillingMetadata2.json()]
a = json.dumps(wyn)

# ODWRÓCENIE WCZESNIEJSZEJ OPERACJI
a_list = json.loads(a)

#a_list[1]

In [ ]:
print(datetime.date.today())

In [ ]:
# Potem sie z tego robi DF w taki sposob, tutaj tez trzeba bedzie napisac for loop gdzie bedzie przechodzic po kazdym elemencie z listy a_list i 
# dodawac wiersze do DF dla kolejnych cik
allForms = pd.DataFrame.from_dict(fillingMetadata.json()["filings"]["recent"])
allForms.head()

In [ ]:
ingest_time = str(datetime.datetime.now(ZoneInfo("Europe/Warsaw")))

In [ ]:
#fillingMetadata.json()

In [ ]:
type(json.dumps(fillingMetadata.json()))

In [ ]:
cik

In [ ]:
#fillingMetadata.json()

In [ ]:
ingest_time

In [ ]:
#fillingMetadata2.json()

In [ ]:
# ZAPIS DO BRONZE POPRAWNY 
blob = bucket.blob(f"company_forms/sec_metadata_folder/{datetime.date.today()}/{cik}_metadata_file_{ingest_time}")
blob.upload_from_string(json.dumps(fillingMetadata.json()), content_type = "application/json")

In [ ]:
cik2

In [ ]:
#fillingMetadata2.json()

In [ ]:
# ZAPIS DO BRONZE POPRAWNY 
blob = bucket.blob(f"company_forms/sec_metadata_folder/{datetime.date.today()}/{cik2}_metadata_file_{ingest_time}")
blob.upload_from_string(json.dumps(fillingMetadata2.json()), content_type = "application/json")

In [ ]:
    # 2.3 XBRL dane

In [ ]:
companyFacts = requests.get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json", headers = headers)
companyFacts2 = requests.get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik2}.json", headers = headers)

In [ ]:
'''
wyn_xbrl = [companyFacts.json(),
            companyFacts2.json()
]

a = json.dumps(wyn_xbrl)
'''

In [ ]:
# ZAPIS XBRL do BRONZE POPRAWNE
blob = bucket.blob(f"company_forms/sec_facts_folder/{datetime.date.today()}/{cik}_fact_file_{ingest_time}")
blob.upload_from_string(json.dumps(companyFacts.json()), content_type = "application/json")

In [ ]:
# ZAPIS XBRL do BRONZE POPRAWNE
blob = bucket.blob(f"company_forms/sec_facts_folder/{datetime.date.today()}/{cik2}_fact_file_{ingest_time}")
blob.upload_from_string(json.dumps(companyFacts2.json()), content_type = "application/json")

In [ ]:
#companyFacts.json()

In [ ]:
    ### 2.4 ALPHA VANTAGE 

In [ ]:
alpha_vantage_1 = requests.get(f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={TICKER}&outputsize=compact&apikey={API_KEY}")
alpha_vantage_2 = requests.get(f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={TICKER_2}&outputsize=compact&apikey={API_KEY}")
#alpha_vantage.json()

In [ ]:
alpha_vantage_2.json()

In [ ]:
wyn_alpha_vantage = [alpha_vantage_1.json(), alpha_vantage_2.json()]

a = json.dumps(wyn_alpha_vantage)
#

In [ ]:
#a

In [ ]:
# ZAPIS ALPHA VANTAGE DO BRONZE 

In [ ]:
blob = bucket.blob(f"alpha_vantage/{str(datetime.datetime.now(ZoneInfo("Europe/Warsaw")))}")
blob.upload_from_string(json.dumps(wyn_alpha_vantage), content_type = "application/json")